In [1]:
# PARAMETERS
img_path = "../../datasetCocoTrainTest/test/z5388400836796_3e44d652d3c9ec5c0d5005d2fd19d37e_jpg.rf.2a8ce04eeff6c63bfa92704f28827802.jpg"  # default, will be overwritten
gradcam_output = None

> ##### Import library

In [2]:
# from ultralytics import YOLO
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

> ##### **Visualize Backpropagation Gradient**
**Load trained model**

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("FasterRCNN/best.pt", map_location=device)
model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 5)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

C:\Users\louis\AppData\Local\Temp\ipykernel_8396\3030266472.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../../FasterRCNN/best.pt", map_locat

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu

**preprocess for Grad-Cam**

In [4]:
def preprocess_image(img_path, img_size=640):
    img = cv2.imread(img_path)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (img_size, img_size))
    t = torch.from_numpy(rgb).permute(2,0,1).float()/255.0
    t = t.unsqueeze(0).to(device)
    return img, t  # BGR original (สำหรับ overlay), และ tensor

**Hook for Grad-CAM**

capture the output and gradients of the last convolutional layer.

* remove hook before register_hook

In [5]:
def register_gradcam_hooks(target_layer):
    activations = {}
    gradients = {}
    def forward_hook(module, input, output):
        activations['value'] = output

    def backward_hook(module, grad_in, grad_out):
        gradients['value'] = grad_out[0]

    fwd_handle = target_layer.register_forward_hook(forward_hook)
    bwd_handle = target_layer.register_backward_hook(backward_hook)

    return activations, gradients, (fwd_handle, bwd_handle)


**Calculate Grad-CAM**

In [6]:
@torch.no_grad()
def forward_no_grad(images_tensor):
    # *** ไม่มี gradient ***
    # model.transform จัดการภาพ (resize/normalize) -> Images object
    images, _ = model.transform(images_tensor)
    # backbone features (dict[str, Tensor]) เพราะ FPN
    features = model.backbone(images.tensors)
    if isinstance(features, torch.Tensor):
        features = {"0": features}
    # สร้าง proposals จาก RPN
    proposals, _ = model.rpn(images, features)
    # RoI-Align (box_roi_pool) เพื่อดึง feature ต่อ proposal
    box_feats = model.roi_heads.box_roi_pool(features, proposals, images.image_sizes)
    box_feats = model.roi_heads.box_head(box_feats)
    class_logits, box_regression = model.roi_heads.box_predictor(box_feats)
    return images, features, proposals, class_logits, box_regression

def forward_with_grad(images_tensor, target_layer, activations, gradients):
    """
    เวอร์ชันมี gradient (อย่าใช้ @torch.no_grad) เพื่อรองรับ backward
    """
    images, _ = model.transform(images_tensor.requires_grad_(True))
    features = model.backbone(images.tensors)
    if isinstance(features, torch.Tensor):
        features = {"0": features}
    proposals, _ = model.rpn(images, features)
    # ผ่าน roi_heads ด้วยตัวเองเพื่อให้มีกราฟครบ
    box_feats = model.roi_heads.box_roi_pool(features, proposals, images.image_sizes)
    box_feats = model.roi_heads.box_head(box_feats)
    class_logits, box_regression = model.roi_heads.box_predictor(box_feats)
    return images, features, proposals, class_logits, box_regression


In [7]:
def select_top_boxes_per_class(class_logits, proposals, image_index=0, class_range=(1,5), k=1):
    """
    class_logits: [N, num_classes] (รวม background class ที่ index=0 ของ torchvision)
    proposals: list[Tensor] ความยาว batch; proposals[0] คือ Tensor [N,4] หลัง RPN (ก่อน NMS ของ roi_heads)
    class_range: (start, end) สำหรับคลาสที่สนใจ (ไม่รวม end) เช่น (1,5) => คลาส 1..4
    k: หยิบ top-k ต่อคลาส
    Return: list of dict { 'idx': i, 'cls': c, 'score': score, 'box': xyxy }
    """
    N, C = class_logits.shape
    probs = F.softmax(class_logits, dim=1)  # [N,C]
    boxes = proposals[image_index]          # [N,4]

    picks = []
    start, end = class_range
    end = min(end, C)  # กันหลุดขอบ
    for c in range(start, end):
        cls_scores = probs[:, c]  # [N]
        topk = torch.topk(cls_scores, k=k)
        for score, idx in zip(topk.values.tolist(), topk.indices.tolist()):
            picks.append({
                "idx": idx,
                "cls": c,
                "score": float(score),
                "box": boxes[idx].detach().cpu().tolist()
            })
    return picks


In [12]:
def overlay_heatmap(bgr_img, heatmap, alpha=0.5):
    return cv2.addWeighted(bgr_img, alpha, heatmap, 1-alpha, 0)

def compute_gradcam_for_pick(images_tensor, pick, activations, gradients, orig_shape):
    # เคลียร์ grad
    model.zero_grad()

    # รัน forward พร้อมกราฟ
    images, features, proposals, class_logits, box_regression = forward_with_grad(images_tensor, None, activations, gradients)

    # ดึง scalar logit ของ box i, class c (ใช้ logit ดีกว่า score เพื่อสัญญาณนิ่งกว่า)
    i = pick["idx"]
    c = pick["cls"]
    score_tensor = class_logits[i, c]

    # backward จาก scalar เดียว
    score_tensor.backward(retain_graph=True)

    # ดึง activation/gradient ของเลเยอร์ที่ hook
    grad = gradients['value']       # [B,C,H,W] หรือ [C,H,W] แล้วแต่เลเยอร์
    act  = activations['value']
    if grad.dim() == 3:  # [C,H,W]
        grad = grad.unsqueeze(0)
        act  = act.unsqueeze(0)

    # ใช้ตัวอย่างแรกของ batch
    grad = grad[0]  # [C,H,W]
    act  = act[0]   # [C,H,W]

    # Global average pooling => weights
    weights = grad.mean(dim=(1,2), keepdim=True)  # [C,1,1]

    cam = (weights * act).sum(dim=0)  # [H,W]
    cam = torch.relu(cam)
    cam = cam / (cam.max() + 1e-8)
    cam_np = cam.detach().cpu().numpy()

    H, W = orig_shape[:2]
    cam_np = cv2.resize(cam_np, (W, H))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_np), cv2.COLORMAP_JET)
    return heatmap


In [9]:
def get_layer(layer_target):
    """
    layer_target:
      - "layer4": ใช้ทั้งบล็อก
      - ("layer4", 2, "conv3"): ระบุถึง submodule
    """
    body = model.backbone.body
    if isinstance(layer_target, str):
        return getattr(body, layer_target)
    elif isinstance(layer_target, (tuple, list)):
        cur = body
        for k in layer_target:
            cur = cur[k] if isinstance(k, int) else getattr(cur, k)
        return cur
    else:
        raise ValueError("layer_target ต้องเป็น str หรือ tuple/list")


In [ ]:
def get_grad_cam_fasterrcnn(img_path, layer_target=("layer4", 2, "conv3"),
                            class_indices=(1,5), k_per_class=1,
                            out_dir="fasterrcnn_gradcam_out", class_names=None):
    """
    layer_target: str เช่น "layer4" หรือ tuple เช่น ("layer4", 2, "conv3")
    class_indices: (start, end) คลาสที่สนใจ (COCO background=0, ตั้งแต่ 1 เป็นต้นไป)
    k_per_class: เลือก top-k ต่อคลาส
    class_names: list ชื่อคลาสตาม index (ถ้ามี)
    """
    os.makedirs(out_dir, exist_ok=True)
    target_layer = get_layer(layer_target)
    activations, gradients, handles = register_gradcam_hooks(target_layer)

    # เตรียมรูป
    bgr_img, img_tensor = preprocess_image(img_path, img_size=640)

    # รัน forward (no-grad) เพื่อหา candidates เร็ว ๆ
    with torch.no_grad():
        images, features, proposals, class_logits, box_regression = forward_no_grad(img_tensor)

    # เลือก box ต่อคลาส
    picks = select_top_boxes_per_class(class_logits, proposals, image_index=0,
                                       class_range=class_indices, k=k_per_class)

    results = []
    for pick in picks:
        # คำนวณ Grad-CAM ของ pick นี้
        heatmap = compute_gradcam_for_pick(img_tensor, pick, activations, gradients, bgr_img.shape)
        overlay = overlay_heatmap(bgr_img, heatmap, alpha=0.5)

        cls_id = pick["cls"]
        cls_name = class_names[cls_id] if (class_names is not None and cls_id < len(class_names)) else f"cls_{cls_id}"
        layer_name = "-".join(map(str, layer_target)) if isinstance(layer_target, (tuple, list)) else str(layer_target)

        save_dir = os.path.join(out_dir, cls_name)
        os.makedirs(save_dir, exist_ok=True)
        # save_path = os.path.join(save_dir, f"{layer_name}_idx{pick['idx']}.jpg")
        # YOLOv8/app_result/gradcam_output/{box["class"]}/{layer_name}.jpg'
        save_path = os.path.join(save_dir, f"{layer_name}.jpg")
        cv2.imwrite(save_path, overlay)

        results.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "score": pick["score"],
            "box_xyxy": pick["box"],
            "save_path": save_path
        })

    # ถอด hooks
    for h in handles:
        h.remove()

    return results


**Grad-Cam for Post-Processed box**

experiment

**Use Grad-CAM on test image**

In [ ]:
# layers = [3, (12, 'cv2'), (18, 'cv1')]
# output = []
# for i, layer in enumerate(layers):
#     output_images, boxes = get_grad_cam(img_path, layer)
#     output.append(output_images)
    
# gradcam_output = output

layers = [
        ("layer2", 1, "conv3"),
        ("layer3", 2, "conv3"),
        ("layer4", 2, "conv3")
]
class_names = ["__background__", "1", "10", "2", "5"]
output = []
for i, layer in enumerate(layers):
    results = get_grad_cam_fasterrcnn(
        img_path,
        layer_target=layer,
        class_indices=(1,5),
        k_per_class=1,
        out_dir="FasterRCNN/app_result/gradcam_output",
        # YOLOv8/app_result/gradcam_output/{box["class"]}/{layer_name}.jpg'
        class_names=class_names
    )
    output.extend(results)
gradcam_output = output
    # print(f"Results for layer {layer}:")
    # for r in results:
    #     print(r)


In [14]:
gradcam_output

[{'class_id': 1,
  'class_name': '1',
  'score': 0.002338883699849248,
  'box_xyxy': [275.370849609375,
   465.7698059082031,
   511.1151123046875,
   583.3374633789062],
  'save_path': 'FasterRCNN/app_result/gradcam_output\\1\\layer2-1-conv3_idx17.jpg'},
 {'class_id': 2,
  'class_name': '10',
  'score': 0.0005399977671913803,
  'box_xyxy': [255.9742889404297,
   409.26629638671875,
   413.8740234375,
   576.4781494140625],
  'save_path': 'FasterRCNN/app_result/gradcam_output\\10\\layer2-1-conv3_idx15.jpg'},
 {'class_id': 3,
  'class_name': '2',
  'score': 0.007557186298072338,
  'box_xyxy': [186.0968475341797,
   291.1148681640625,
   520.570068359375,
   595.0645751953125],
  'save_path': 'FasterRCNN/app_result/gradcam_output\\2\\layer2-1-conv3_idx100.jpg'},
 {'class_id': 4,
  'class_name': '5',
  'score': 0.9981350898742676,
  'box_xyxy': [252.30311584472656,
   387.0874938964844,
   513.3898315429688,
   590.54345703125],
  'save_path': 'FasterRCNN/app_result/gradcam_output\\5\\lay